# Image Basics

This short tutorial introduces images as data. You will need this foundation for everything else in the course — and for your project.

There is nothing to fill in. Read through each section, run the code, and make sure you understand the output before moving on. The whole thing should take about 30 minutes.

**What you will learn:**
- What a digital image actually is (a numpy array)
- What colour channels are and why they matter
- How pixel values are represented and why the range matters
- How to load, display, and do basic operations on images
- What colour spaces are and when to use them

## Setup

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Download a sample image to work with
import urllib.request
url = 'https://raw.githubusercontent.com/klaverhenrik/Deep-Learning-for-Visual-Recognition-2026/main/data/Cute_dog.jpg'
urllib.request.urlretrieve(url, 'dog.jpg')
print('Image downloaded.')

## 1. An image is a numpy array

A digital image is just a grid of numbers. Each number represents the brightness 
of one pixel. For a colour image, there are three grids stacked together — one 
for each colour channel (more on this in section 2).

Let's load an image and look at its structure.

In [ ]:
# Load the image using OpenCV
img = cv2.imread('dog.jpg')

# What type of object is it?
print(f'Type:  {type(img)}')

# What is its shape?
print(f'Shape: {img.shape}')   # (height, width, channels)

# What data type are the pixel values?
print(f'Dtype: {img.dtype}')   # uint8 = unsigned 8-bit integer

# What is the range of pixel values?
print(f'Min pixel value: {img.min()}')
print(f'Max pixel value: {img.max()}')

The shape `(height, width, 3)` tells us this is a colour image with 3 channels.
The dtype `uint8` means pixel values are integers in the range **0 to 255**.

This 0–255 range is the standard for images stored on disk. However, neural networks 
expect values in a different range — typically **0 to 1** or **-1 to 1**. 
Converting between these ranges is called **normalisation**, and we will do it 
every time we feed images into a model.

Let's look at the actual pixel values:

In [ ]:
# Print the pixel values in the top-left 5x5 region of the image
# Each pixel has 3 values — one per colour channel
print('Top-left 5x5 pixels (each row is one pixel, each column is one channel):')
print(img[:5, :5, :])   # [rows, columns, channels]

## 2. Colour channels

A colour image has three channels: **Red**, **Green**, and **Blue** (RGB).
Each channel is a 2D grid of values representing the intensity of that colour 
at each pixel position.

**Important:** OpenCV loads images in **BGR** order (Blue, Green, Red) — 
the reverse of the standard RGB order used by most other libraries, including 
matplotlib and PyTorch. This is a common source of bugs: if you load with OpenCV 
and display with matplotlib without converting, the colours will look wrong.

In [ ]:
# Display the image loaded directly with OpenCV (BGR order)
# Notice that the colours look wrong — reds appear blue and vice versa
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.imshow(img)   # matplotlib expects RGB, but img is BGR
plt.title('Loaded with OpenCV, displayed as-is\n(colours are wrong)')
plt.axis('off')

# Convert from BGR to RGB before displaying
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.subplot(1, 2, 2)
plt.imshow(img_rgb)   # now correct
plt.title('After converting BGR → RGB\n(colours are correct)')
plt.axis('off')

plt.tight_layout()
plt.show()

print('Rule: always convert BGR → RGB before displaying with matplotlib.')

In [ ]:
# Let's look at the three channels separately
# Remember: img is BGR, so channel 0 = Blue, 1 = Green, 2 = Red
b_channel = img[:, :, 0]   # Blue
g_channel = img[:, :, 1]   # Green
r_channel = img[:, :, 2]   # Red

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(img_rgb)
axes[0].set_title('Original (RGB)')
axes[0].axis('off')

axes[1].imshow(r_channel, cmap='Reds')
axes[1].set_title('Red channel')
axes[1].axis('off')

axes[2].imshow(g_channel, cmap='Greens')
axes[2].set_title('Green channel')
axes[2].axis('off')

axes[3].imshow(b_channel, cmap='Blues')
axes[3].set_title('Blue channel')
axes[3].axis('off')

plt.suptitle('Each colour channel is a 2D grid of values (0–255)', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Shape of one channel: {r_channel.shape}')   # (height, width) — no channel dimension

## 3. Grayscale images

A grayscale image has only one channel — just brightness, no colour information. 
It has shape `(height, width)` rather than `(height, width, 3)`.

Many image processing operations are easier to understand in grayscale, and some 
models expect grayscale input (e.g. models trained on medical images or documents).

In [ ]:
# Convert to grayscale
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

print(f'Colour image shape:    {img.shape}')       # (H, W, 3)
print(f'Grayscale image shape: {img_gray.shape}')  # (H, W)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title('Colour (RGB)')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img_gray, cmap='gray')   # cmap='gray' is important for single-channel images
plt.title('Grayscale')
plt.axis('off')

plt.tight_layout()
plt.show()

## 4. Pixel value ranges and normalisation

Images on disk use integer values from **0 to 255** (`uint8`).
Neural networks work much better with floating point values in a smaller range.

There are two common normalisation conventions you will encounter:

| Convention | Range | When used |
|---|---|---|
| Divide by 255 | 0.0 to 1.0 | General purpose |
| ImageNet normalisation | ~-2.1 to ~2.6 | Pretrained models (ResNet, ViT, etc.) |

The ImageNet normalisation subtracts the mean and divides by the standard deviation 
of the ImageNet dataset — this is what `torchvision.transforms.Normalize` does, 
and it's what we used in the Lab 5 training pipeline.

In [ ]:
# Original: uint8, range 0–255
print(f'Original dtype:  {img.dtype}')
print(f'Original range:  {img.min()} – {img.max()}')

# Convert to float and normalise to 0–1
img_float = img_rgb.astype(np.float32) / 255.0
print(f'\nNormalised dtype:  {img_float.dtype}')
print(f'Normalised range:  {img_float.min():.3f} – {img_float.max():.3f}')

# ImageNet normalisation (what torchvision.transforms.Normalize does)
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])
img_imagenet = (img_float - mean) / std
print(f'\nImageNet normalised range:  {img_imagenet.min():.3f} – {img_imagenet.max():.3f}')

# Important: you cannot display an ImageNet-normalised image directly
# You must reverse the normalisation first (this is what denormalise() does in Lab 5)

## 5. Basic image operations

Because images are numpy arrays, you can do anything to them that you can do 
to a numpy array. Here are the operations you will use most often in your project.

In [ ]:
# --- Resizing ---
# Neural networks require all images to be the same size.
# Resizing is almost always the first preprocessing step.

img_small  = cv2.resize(img_rgb, (64, 64))    # resize to 64x64
img_medium = cv2.resize(img_rgb, (224, 224))  # resize to 224x224 (standard for ImageNet models)

print(f'Original size: {img_rgb.shape}')
print(f'Resized 64x64: {img_small.shape}')
print(f'Resized 224x224: {img_medium.shape}')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, image, title in zip(axes,
                             [img_rgb, img_small, img_medium],
                             ['Original', '64x64', '224x224']):
    ax.imshow(image)
    ax.set_title(f'{title}\n{image.shape[0]}x{image.shape[1]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# --- Cropping ---
# Cropping is just numpy array slicing: img[row_start:row_end, col_start:col_end]
# Remember: the first axis is rows (height), the second is columns (width)

h, w = img_rgb.shape[:2]

# Crop the centre of the image
centre_crop = img_rgb[h//4 : 3*h//4,
                       w//4 : 3*w//4]

# Crop the top-left quadrant
top_left = img_rgb[:h//2, :w//2]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, image, title in zip(axes,
                             [img_rgb, centre_crop, top_left],
                             ['Original', 'Centre crop', 'Top-left quadrant']):
    ax.imshow(image)
    ax.set_title(f'{title}\n{image.shape[0]}x{image.shape[1]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# --- Flipping ---
# Horizontal flipping is the most common data augmentation technique.
# It doubles your effective dataset size for free.

img_flipped_h = cv2.flip(img_rgb, 1)   # 1 = horizontal flip
img_flipped_v = cv2.flip(img_rgb, 0)   # 0 = vertical flip

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, image, title in zip(axes,
                             [img_rgb, img_flipped_h, img_flipped_v],
                             ['Original', 'Horizontal flip', 'Vertical flip']):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Colour spaces

RGB is not the only way to represent colour. Different colour spaces separate 
colour information in different ways, which can be useful for different tasks.

The two most useful alternatives to RGB are:

**HSV (Hue, Saturation, Value)**  
Separates the colour itself (hue) from how vivid it is (saturation) and how bright 
it is (value). Useful when you want to detect or filter by colour regardless of lighting.

**LAB (Lightness, A, B)**  
Separates brightness (L) from colour (A and B channels). Designed to match human 
colour perception. Often used in medical imaging and style transfer.

In [ ]:
# Convert to HSV and LAB
img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
img_lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

# Row 1: RGB
axes[0, 0].imshow(img_rgb)
axes[0, 0].set_title('RGB')
for i, (ch, name) in enumerate(zip(cv2.split(img_rgb), ['R', 'G', 'B'])):
    axes[0, i+1].imshow(ch, cmap='gray')
    axes[0, i+1].set_title(f'RGB — {name} channel')

# Row 2: HSV
axes[1, 0].imshow(cv2.cvtColor(img_hsv, cv2.COLOR_HSV2RGB))
axes[1, 0].set_title('HSV (converted back for display)')
for i, (ch, name) in enumerate(zip(cv2.split(img_hsv), ['Hue', 'Saturation', 'Value'])):
    axes[1, i+1].imshow(ch, cmap='gray')
    axes[1, i+1].set_title(f'HSV — {name}')

# Row 3: LAB
axes[2, 0].imshow(cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB))
axes[2, 0].set_title('LAB (converted back for display)')
for i, (ch, name) in enumerate(zip(cv2.split(img_lab), ['L (Lightness)', 'A (green–red)', 'B (blue–yellow)'])):
    axes[2, i+1].imshow(ch, cmap='gray')
    axes[2, i+1].set_title(f'LAB — {name}')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Three colour spaces and their channels', fontsize=13)
plt.tight_layout()
plt.show()

## 7. What neural networks actually see

When you feed an image into a PyTorch model, it needs to be a tensor 
with a specific shape and value range. Let's see what that looks like.

PyTorch uses the convention **(C, H, W)** — channels first, then height and width. 
This is the opposite of numpy, which uses **(H, W, C)**. The conversion is done 
automatically by `torchvision.transforms.ToTensor()`.

In [ ]:
import torch
from torchvision import transforms
from PIL import Image

# Load the image with PIL (torchvision transforms expect PIL images)
pil_image = Image.open('dog.jpg').convert('RGB')

# Define a simple transform pipeline (same as what we use in Lab 5)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),                          # converts to (C, H, W), range 0–1
    transforms.Normalize(                           # ImageNet normalisation
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

tensor = transform(pil_image)

print(f'PIL image size:     {pil_image.size}')   # (width, height) — PIL convention
print(f'Tensor shape:       {tensor.shape}')     # (C, H, W) — PyTorch convention
print(f'Tensor dtype:       {tensor.dtype}')
print(f'Tensor value range: {tensor.min():.3f} – {tensor.max():.3f}')

# To add a batch dimension (models expect batches, not single images):
batch = tensor.unsqueeze(0)   # (1, C, H, W)
print(f'\nWith batch dimension: {batch.shape}')

## Summary

Here is everything you need to remember:

| Concept | What to remember |
|---|---|
| Image = numpy array | Shape is `(H, W, C)` for colour, `(H, W)` for grayscale |
| Pixel values | Integers 0–255 on disk; floats 0–1 or ImageNet-normalised for models |
| OpenCV colour order | BGR, not RGB — always convert before displaying with matplotlib |
| PyTorch tensor shape | `(C, H, W)` — channels first, opposite of numpy |
| Normalisation | Always use the same normalisation as the pretrained model was trained with |
| Resizing | All images in a batch must be the same size |

The most common bugs in image deep learning projects are:
1. Forgetting to convert BGR → RGB when loading with OpenCV
2. Using the wrong normalisation for a pretrained model
3. Forgetting that PyTorch tensors are `(C, H, W)` while numpy arrays are `(H, W, C)`

You will probably make all three of these mistakes at some point. Now you know 
what to look for when something looks wrong.

---

**You are done with this tutorial.** Move on to the demo notebooks to see what 
deep learning can actually do with images.